# 🎛️ Fine-tuning Guide 
[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/togethercomputer/together-cookbook/blob/main/fine-tuning/Finetuning_Guide.ipynb)

Large Language Models (LLMs) offer powerful general capabilities, but often require **fine-tuning** to excel at specific tasks or understand domain-specific language. Fine-tuning adapts a trained model to a smaller, targeted dataset, enhancing its performance for your unique needs.

This notebook provides a step-by-step guide to fine-tuning models using the Together AI platform. We will walk through the entire process, from preparing your data to evaluating your fine-tuned model.

We will cover:

1.  **Dataset Preparation:** Loading a standard dataset, transforming it into the required format for supervised fine-tuning on Together AI, and uploading your formatted dataset to Together AI Files.
2.  **Fine-tuning Job Launch:** Configuring and initiating a fine-tuning job using the Together AI API.
3.  **Job Monitoring:** Checking the status and progress of your fine-tuning job.
4.  **Deployment and Inference:** Deploying your fine-tuned model on a dedicated endpoint and querying it.
5.  **Evaluation:** Comparing the performance of the finetuned model against the base model on a test set.
6.  **Clean Up:** Stopping the endpoint so it stops billing.

By following this guide, you'll gain practical experience in creating specialized LLMs tailored to your specific requirements using Together AI.

## Setup and Installation
---
First, install the necessary Python libraries. We need:
- `together`: The official Together AI Python client for interacting with the API (fine-tuning, inference, files, etc.).
- `datasets`: A library from Hugging Face for easily downloading and manipulating datasets.
- `tqdm`: To enable interactive elements like progress bars within the notebook.

In [1]:
!pip install -qU together "datasets>=5.0.1" "tqdm>=4.66.3"

## 1. Dataset Preparation
---
Fine-tuning requires data formatted in a specific way. We'll use the a conversational dataset as an example - here the goal of the fine-tuning is to improve the model on multi-turn conversations. 

First we need to transform this dataset into the chat format expected by Together AI for supervised fine-tuning.

The required format is a JSON object per line, where each object contains a list of conversation turns under the `"messages"` key.

Each message must have a `"role"` (`system`, `user`, or `assistant`) and `"content"`.

Conversation Data Example:
```json
{"messages": [{"role": "system", "content": "You are a helpful assistant."}, 
              {"role": "user", "content": "Hello!"}, 
              {"role": "assistant", "content": "Hi! How can I help you?"}]}
```

🔗Depending on what type of fine-tuning you want to perform you can also pass in [instruction data](https://docs.together.ai/docs/fine-tuning-data-preparation#instruction-data), [preference data](https://docs.together.ai/docs/fine-tuning-data-preparation#preference-data) or even [simple text data](https://docs.together.ai/docs/fine-tuning-data-preparation#generic-text-data).

### Load Raw Dataset
We use the `datasets` library to download the CoQA dataset from the Hugging Face Hub.

Let's examine the structure of the raw dataset. CoQA provides a story, a series of questions related to the story, and corresponding answers.

In [2]:
from datasets import load_dataset

coqa_dataset = load_dataset("stanfordnlp/coqa")

In [3]:
coqa_dataset["train"].to_pandas().head()

,source,story,questions,answers
0,wikipedia,"The Vatican Apostolic Library (), more commonl...","[When was the Vat formally opened?, what is th...",{'input_text': ['It was formally established i...
1,cnn,New York (CNN) -- More than 80 Michael Jackson...,"[Where was the Auction held?, How much did the...","{'input_text': ['Hard Rock Cafe', '$2 million...."
2,gutenberg,"CHAPTER VII. THE DAUGHTER OF WITHERSTEEN \n\n""...","[What did Venters call Lassiter?, Who asked La...","{'input_text': ['gun-man', 'Jane', 'Yes', 'to ..."
3,cnn,(CNN) -- The longest-running holiday special s...,"[Who is Rudolph's father?, Why does Rudolph ru...","{'input_text': ['Donner', 'he felt like an out..."
4,gutenberg,CHAPTER XXIV. THE INTERRUPTED MASS \n\nThe mor...,"[Who arrived at the church?, Who was followed ...","{'input_text': ['the garrison first', 'Fra. Do..."


### Transform Data to Chat Format

Now, we need to convert each row of the CoQA dataset into the required chat format (`[{'role': ..., 'content': ...}, ...]`).

We'll create a function `map_coqa_to_chat_format` that takes a row from the dataset and structures it as a conversation:
1.  A `system` message containing the story (context).
2.  Alternating `user` (question) and `assistant` (answer) messages.


In [4]:
# the system prompt,if present, must always be at the beginning
system_prompt = "Read the story and extract answers for the questions.\nStory: {}"

def map_fields(row):
    """    
    Maps the fields from a row of data to a structured format for conversation.
    Args:
        row (dict): A dictionary containing the keys "story", "questions", and "answers".
            - "story" (str): The story content to be used in the system prompt.
            - "questions" (list of str): A list of questions from the user.
            - "answers" (dict): A dictionary containing the key "input_text" which is a list of answers from the assistant.
    Returns:
        dict: A dictionary with a single key "messages" which is a list of message dictionaries.
            Each message dictionary contains:
            - "role" (str): The role of the message sender, either "system", "user", or "assistant".
            - "content" (str): The content of the message.    
    """
    # create system prompt
    messages = [{"role": "system", "content": system_prompt.format(row["story"])}]
    
    # add user and assistant messages
    for q, a in zip(row["questions"], row["answers"]["input_text"]):
        messages.append({"role": "user", "content": q})
        messages.append({"role": "assistant", "content": a})
    
    return {"messages": messages}

We apply this transformation function to the entire dataset using the `.map()` method. We also remove the original columns as they are no longer needed after transformation.

In [5]:
# transform the data using the mapping function

train_messages = coqa_dataset["train"].map(map_fields, remove_columns=coqa_dataset["train"].column_names)

Let's check the structure of our transformed dataset. It should now only contain the `messages` column.

Here's an example of a single processed data point:

In [6]:
train_messages

Dataset({
    features: ['messages'],
    num_rows: 7199
})

Write the dataset out to a `json` file:

In [7]:
train_messages.to_json("coqa_prepared_train.jsonl")

Creating json from Arrow format:   0%|          | 0/8 [00:00<?, ?ba/s]

23777505

### Upload Data to Together AI

Now that we have our formatted `coqa_prepared_train.jsonl` files, we need to check if they meet the format specification and then upload them to Together AI. Fine-tuning jobs read data directly from your uploaded files.

We use the `check_file` function to check the file and `files.upload()` method. This returns information about the uploaded file, including its ID, which we'll need later to start the fine-tuning job.

In [8]:
## Setup Together AI client
from together import Together
import os
import json

TOGETHER_API_KEY = os.getenv("TOGETHER_API_KEY")
WANDB_API_KEY = os.getenv("WANDB_API_KEY") # needed for logging fine-tuning to wandb


client = Together(api_key=TOGETHER_API_KEY)

In [9]:
# The v2 SDK validates fine-tuning files at upload time (check=True is the
# default for client.files.upload), so a malformed dataset errors before any
# job is created. No separate pre-upload check step is needed.
print("Files will be validated during upload.")


{
  "is_check_passed": true,
  "message": "Checks passed",
  "found": true,
  "file_size": 23777505,
  "utf8": true,
  "line_type": true,
  "text_field": true,
  "key_value": true,
  "has_min_samples": true,
  "num_samples": 7199,
  "load_json": true,
  "filetype": "jsonl"
}


In [10]:
## Upload the data to Together

train_file_resp = client.files.upload("coqa_prepared_train.jsonl", check=True)
print(f"Train file response: {train_file_resp.id}")

Uploading file coqa_prepared_train.jsonl: 100%|██████████| 23.8M/23.8M [00:01<00:00, 15.7MB/s]


Train file response: file-9554964d-5711-419a-bcc2-c4edaaa07ee3


## 2. Launch Fine-tuning Job
---
With our data uploaded, we can now launch the fine-tuning job using `client.fine_tuning.create()`.

Key parameters:
- `model`: The base model you want to finetune (e.g., `'togethercomputer/llama-2-7b-chat'`). Choose from the models available for fine-tuning on Together AI.
- `training_file`: The ID of your uploaded training JSONL file.
- `validation_file`: The ID of your uploaded validation JSONL file (optional, but highly recommended for monitoring).
- `suffix`: A custom string added to the base model name to create your unique finetuned model name (e.g., `my-coqa-ft`). Keep it short and descriptive.
- `n_epochs`: The number of times the model will see the entire training dataset.
- `n_checkpoints`: Number of checkpoints to save during training (useful for resuming or selecting the best model). Set to 1 if you only need the final model.
- `learning_rate`: Controls how much the model weights are updated during training. Needs tuning.
- `batch_size`: Number of training examples processed in one iteration. Depends on model size and available resources.

🔗 For an exhaustive list of all the available fine-tuning parameters refer to the [Together AI Fine-tuning API Reference](https://docs.together.ai/reference/post_fine-tunes) docs.

🔗 For a list of all the models you can fine-tune on the Together AI platform see [docs here](https://docs.together.ai/docs/fine-tuning-models).

In [ ]:
## This fine-tuning job should take ~10-15 minutes to complete

ft_resp = client.fine_tuning.create(
    training_file = train_file_resp.id,
    model = 'meta-llama/Meta-Llama-3.1-8B-Instruct-Reference',
    train_on_inputs= "auto",
    n_epochs = 3,
    n_checkpoints = 1,
    wandb_api_key = WANDB_API_KEY,
    lora = True,
    warmup_ratio=0,
    learning_rate = 1e-5,
    suffix = 'test1_8b',
)

print(ft_resp.id)

## 3. Monitor Fine-tuning Job
---
Fine-tuning can take time depending on the model size, dataset size, and hyperparameters. You can monitor and alter the job's progress using the following methods:

- List all jobs: `client.fine_tuning.list()`
- Status of a Job:`client.fine_tuning.retrieve(id=ft_resp.id)`
- List all events for a Job: `client.fine_tuning.list_events(id=ft_resp.id)`: Retrieves logs and events generated during the job
- Cancel job: `client.fine_tuning.cancel(id=ft_resp.id)`
- Download model after done: `client.fine_tuning.content(ft_id=ft_resp.id)`

Once the job is complete (`status == 'completed'`), the response from `retrieve` includes the registry identifiers you use to deploy the model in step 4.


In [ ]:
# Check status of the job
resp = client.fine_tuning.retrieve(ft_resp.id)
print(resp.status)

FinetuneJobStatus.STATUS_COMPLETED


In [ ]:
# Wait for the job to complete before continuing (checkpoint must exist)
import time

while True:
    status = client.fine_tuning.retrieve(ft_resp.id)
    print(f"Status: {status.status}")
    if status.status in ["completed", "failed", "cancelled", "error"]:
        break
    time.sleep(60)

assert status.status == "completed", f"Job ended with status {status.status}"


In [ ]:
# this loop will print the logs of the job thus far
for event in client.fine_tuning.list_events(id=ft_resp.id).data:
    print(event.message)

Fine tune request created
Job started at Wed Apr  9 19:48:05 UTC 2025
Model data downloaded for togethercomputer/Meta-Llama-3.1-8B-Instruct-Reference__TOG__FT at Wed Apr  9 19:48:07 UTC 2025
Data downloaded for togethercomputer/Meta-Llama-3.1-8B-Instruct-Reference__TOG__FT at $2025-04-09T19:48:14.918488
WandB run initialized.
Training started for model togethercomputer/Meta-Llama-3.1-8B-Instruct-Reference__TOG__FT
Epoch completed, at step 24
Epoch completed, at step 48
Epoch completed, at step 72
Training completed for togethercomputer/Meta-Llama-3.1-8B-Instruct-Reference__TOG__FT at Wed Apr  9 20:02:24 UTC 2025
Uploading adapter model
Compressing output model
Model compression complete
Uploading output model
Model upload complete
Job finished at Wed Apr  9 20:06:33 UTC 2025


🔗 You can also navigate to the WandB page linked in your [fine-tuning dashboard](https://api.together.ai/fine-tuning) to see the fine-tuning related loss curves and more.

<img src="../images/FT_run.png" width="900">

## 4. Deploy and Query the Fine-tuned Model
---
Fine-tuned models run on [dedicated endpoints](https://docs.together.ai/docs/fine-tuning/deployment). A completed job is already a private model in your project, so there's no upload step. You deploy it, wait for the deployment to become ready, then query it.

⚠️ Dedicated endpoints bill per minute per running replica, even when idle. When you're done, run the clean-up cell in step 6 to stop billing.

To deploy, you need one of the job's registry identifiers for the trained weights:
- `model_object_id` (an `ml_...` value): The SDK deploys by this ID.
- `model_object_name` (a `<project_slug>/<model_name>` value): The CLI accepts this name in place of the ID.

In [ ]:
# Registry identifiers for the trained weights, set once the job is completed.
# The fine-tuned model's output_name isn't deployable, so use one of these instead.
job = client.fine_tuning.retrieve(ft_resp.id)
model_object_id = job.api_model_object_id

print(f"model_object_id:   {model_object_id}")
print(f"model_object_name: {job.api_model_object_name}")

### Option 1: Deploy with the SDK

The SDK deploys in three steps: create the endpoint, bind the model and a config to a deployment on it, then route traffic to that deployment.

First, look up your project ID and a config for the model. A config pins the hardware the deployment runs on (GPU type and count), and a fine-tuned model deploys with a config of its base model.

🔗 For every parameter, see the [endpoints API reference](https://docs.together.ai/reference/dmi/endpoints-create).

In [ ]:
identity = client.whoami()
if identity.type != "project_api_key":
    raise RuntimeError("TOGETHER_API_KEY must be a project API key")
project_id = identity.project.id

# List the deployment configs for the fine-tuned model's base model
ft_model = client.models.retrieve(model_object_id, project_id=project_id)
configs = list(
    client.models.configs.list(
        project_id=project_id,
        reference_model_id=ft_model.base_model_id or ft_model.id,
    )
)
if not configs:
    raise RuntimeError("No deployment configs found for this model's base model")

for cfg in configs:
    print(cfg.id, {s.key: s.value for s in cfg.selectors})

# Pick the config whose hardware you want. Configs can belong to another project,
# so build the path from the config's own project ID.
cfg = configs[0]
config = f"projects/{cfg.project_id}/configs/{cfg.id}"

In [ ]:
ENDPOINT_NAME = f"coqa-{ft_resp.id}"  # must be unique within your project

# 1. Create the endpoint.
endpoint = client.endpoints.create(project_id=project_id, name=ENDPOINT_NAME)

# 2. Bind the model and config to a deployment.
deployment = client.endpoints.deployments.create(
    endpoint.id,
    project_id=project_id,
    name="prod",
    model=f"projects/{project_id}/models/{model_object_id}",
    config=config,
    autoscaling={"min_replicas": 1, "max_replicas": 1},
    inactive_timeout=60,  # stop automatically after 60 idle minutes
)

# 3. Route 100% of traffic to the deployment.
client.endpoints.update(
    endpoint.id,
    project_id=project_id,
    traffic_split=[{"deployment_id": deployment.id, "weight": 1}],
)

# The endpoint string (<project_slug>/<endpoint_name>) is the `model` you pass at inference time
endpoint_string = endpoint.name
print(endpoint_string)

In [ ]:
import time

# Provisioning can take up to 10 minutes
while True:
    deployment = client.endpoints.deployments.retrieve(
        deployment.id, project_id=project_id, endpoint_id=endpoint.id
    )
    state = deployment.status.state
    print(state)
    if state in ("DEPLOYMENT_STATE_READY", "DEPLOYMENT_STATE_FAILED"):
        break
    time.sleep(30)

assert state == "DEPLOYMENT_STATE_READY", f"Deployment ended in {state}"

### Option 2: Deploy with the CLI or Console

Either of these creates the same endpoint. If you deploy this way, set `endpoint_string` to the endpoint's `<project_slug>/<endpoint_name>` value before you query it below.

**CLI:** With Together CLI version `3.0.0` or later (install or upgrade with `uv tool install "together[cli]" --upgrade`), `deploy` creates the endpoint, attaches a deployment, and routes all traffic to it in one step:

```bash
tg endpoints deploy "<MODEL_OBJECT_NAME>" --endpoint coqa-finetune
```

**Console:**
1. Open your job from the [fine-tuning dashboard](https://api.together.ai/fine-tuning), or find the model under [My models](https://api.together.ai/models?category=my-models), and select **Deploy on dedicated**.
2. On the **Create endpoint** page, enter an **Endpoint name**. The console prefixes it with your project slug.
3. Under **Deployment profiles**, choose the hardware to run on, then set **Min replicas** and **Max replicas**.
4. Select **Create endpoint**. The endpoint and its status appear on the [Endpoints dashboard](https://api.together.ai/endpoints).

### Query the Endpoint

Once the deployment is ready, pass the endpoint string as `model` and send the request to `https://api-inference.together.ai`. That base URL serves only inference, so create a second client for it and keep `client` for the fine-tuning and files APIs.

In [ ]:
# A dedicated client for inference: this base URL serves only
# inference, not the fine-tuning or files APIs.
inference_client = Together(api_key=TOGETHER_API_KEY, base_url="https://api-inference.together.ai")

response = inference_client.chat.completions.create(
    model=endpoint_string,
    messages=[{"role": "user", "content": "What is the capital of France?"}],
    max_tokens=128,
)
print(response.choices[0].message.content)

## 5. Evaluation
---
To assess the impact of fine-tuning, we can compare the responses of our finetuned model with the original base model on the same prompt in out test set.

This provides a way to measure improvements, after fine-tuning, to the model's behavior for our specific task (conversational QA based on a story).

In [ ]:
from tqdm.auto import tqdm
from multiprocessing.pool import ThreadPool

# Both models are queried on dedicated endpoints through inference_client.
# The fine-tuned model uses the endpoint from step 4. To compare against the base
# model, deploy meta-llama/Llama-3.1-8B-Instruct on its own dedicated endpoint
# and set base_model to that endpoint string.
base_model = None
finetuned_model = endpoint_string
print(f"Evaluating fine-tuned endpoint: {finetuned_model}")

In [16]:
# We'll load in 50 conversations from the CoQA validation set for evaluation
coqa_dataset_validation = load_dataset("stanfordnlp/coqa", split="validation[:50]")

In [17]:
coqa_dataset_validation

Dataset({
    features: ['source', 'story', 'questions', 'answers'],
    num_rows: 50
})

We'll use the code below to generate answers from the baseline and fine-tuned model.

In [18]:
## This function is used to generate model answers on the CoQA validation set from the untuned reference and fine-tuned models

def get_model_answers(model_name):
    """
    Generate model answers for a given model name using a dataset of questions and answers.
    Args:
        model_name (str): The name of the model to use for generating answers.
    Returns:
        list: A list of lists, where each inner list contains the answers generated by the model for the corresponding set of questions in the dataset.
    The function performs the following steps:
    1. Initializes an empty list to store the model answers.
    2. Defines an inner function `get_answers` that takes a data dictionary and generates answers for the questions in the data.
    3. Uses a thread pool to parallelize the process of generating answers for each entry in the validation dataset.
    4. Appends the generated answers to the `model_answers` list.
    5. Returns the `model_answers` list.
    Note:
        - The `inference_client` variable is assumed to be defined elsewhere in the code.
        - The `coqa_dataset` variable is assumed to contain the dataset with a "validation" key.
    """

    model_answers = []
    system_prompt = "Read the story and extract answers for the questions.\nStory: {}"
    
    def get_answers(data):
        answers = []
        messages = [
            {
                "role": "system",
                "content": system_prompt.format(data["story"]),
            }
        ]
        for q, true_answer in zip(data["questions"], data["answers"]["input_text"]):
            try:
                messages.append(
                    {
                        "role": "user",
                        "content": q
                    }
                )
                response = inference_client.chat.completions.create(
                    messages=messages,
                    model=model_name,
                    max_tokens=64
                )
                answer = response.choices[0].message.content
                answers.append(answer)
            except Exception:
                answers.append("Invalid Response")
        return answers

    # We'll use 8 threads to generate answers faster in parallel
    with ThreadPool(8) as pool:
        for answers in tqdm(pool.imap(get_answers, coqa_dataset_validation), total=len(coqa_dataset_validation)):
            model_answers.append(answers)

    return model_answers

We'll use the functions below to calculate the exact match and F1 score metrics

In [19]:
import collections
import re
import string


# Exact Match and F1 as defined by the official SQuAD evaluation script
def normalize_answer(s):
    """Lowercase, then remove punctuation, articles, and extra whitespace."""
    s = "".join(ch for ch in s.lower() if ch not in string.punctuation)
    s = re.sub(r"\b(a|an|the)\b", " ", s)
    return " ".join(s.split())


def compute_exact(a_gold, a_pred):
    return int(normalize_answer(a_gold) == normalize_answer(a_pred))


def compute_f1(a_gold, a_pred):
    gold_toks = normalize_answer(a_gold).split() if a_gold else []
    pred_toks = normalize_answer(a_pred).split() if a_pred else []
    if not gold_toks or not pred_toks:
        # If either is no-answer, F1 is 1 if they agree and 0 otherwise
        return int(gold_toks == pred_toks)
    num_same = sum((collections.Counter(gold_toks) & collections.Counter(pred_toks)).values())
    if num_same == 0:
        return 0
    precision = num_same / len(pred_toks)
    recall = num_same / len(gold_toks)
    return 2 * precision * recall / (precision + recall)


## This function will be used to evaluate predicted answers using the Exact Match (EM) and F1 metrics
def get_metrics(pred_answers):
    """
    Calculate the Exact Match (EM) and F1 metrics for predicted answers.
    Args:
        pred_answers (list): A list of predicted answers. Each element in the list is a list of predicted answers for a single question.
    Returns:
        tuple: A tuple containing two elements:
            - em_score (float): The average Exact Match score across all predictions.
            - f1_score (float): The average F1 score across all predictions.
    """

    em_metrics = []
    f1_metrics = []

    for pred, data in tqdm(zip(pred_answers, coqa_dataset_validation), total=len(pred_answers)):
        for pred_answer, true_answer in zip(pred, data["answers"]["input_text"]):
            em_metrics.append(compute_exact(true_answer, pred_answer))
            f1_metrics.append(compute_f1(true_answer, pred_answer))

    return sum(em_metrics) / len(em_metrics), sum(f1_metrics) / len(f1_metrics)

In [ ]:
# Runs only if you deployed the base model and set base_model above
if base_model:
    base_answers = get_model_answers(base_model)

In [ ]:
if base_model:
    em_metric, f1_metric = get_metrics(base_answers)
    print(f"Model: Baseline, EM: {em_metric}, F1: {f1_metric}")

In [ ]:
ft_answers = get_model_answers(finetuned_model)

In [ ]:
em_metric, f1_metric = get_metrics(ft_answers)
print(f"Model: {finetuned_model}, \n\nEM: {em_metric}, F1: {f1_metric}")

| Llama 3.1 8B | EM | F1|
|---|---|---|
| Original | 0.01 | 0.18 |
| Fine-tuned | 0.31 | 0.41 |

We can see that the fine-tuned model performs twice as well on the test set when measuring the F1 score.

For a more detailed guide on Fine-tuning follow our [docs here](https://docs.together.ai/docs/finetuning).

## 6. Clean Up
---
The endpoint bills per minute while its deployment runs. The first cell below scales the deployment to zero, which stops billing. The second deletes the deployment and the endpoint once the deployment reaches `DEPLOYMENT_STATE_STOPPED`. To keep the endpoint for later, run only the first cell.

If you deployed with the CLI, `tg endpoints rm "<ENDPOINT_ID>" --force` deletes the endpoint and its deployment in one step. If you deployed from the console, open the endpoint on the [Endpoints dashboard](https://api.together.ai/endpoints) and stop or delete it there. Delete a base-model endpoint the same way if you created one.

In [ ]:
# Scale to zero to stop billing.
client.endpoints.deployments.update(
    deployment.id,
    project_id=project_id,
    endpoint_id=endpoint.id,
    autoscaling={"min_replicas": 0, "max_replicas": 0},
)

In [ ]:
# Wait for the deployment to stop.
while client.endpoints.deployments.retrieve(
    deployment.id, project_id=project_id, endpoint_id=endpoint.id
).status.state != "DEPLOYMENT_STATE_STOPPED":
    time.sleep(15)

# Drop its traffic weight, then delete the deployment and the endpoint.
client.endpoints.update(
    endpoint.id,
    project_id=project_id,
    traffic_split=[{"deployment_id": deployment.id, "weight": 0}],
)
client.endpoints.deployments.delete(
    deployment.id, project_id=project_id, endpoint_id=endpoint.id
)
client.endpoints.delete(endpoint.id, project_id=project_id)
print(f"Deleted {endpoint_string}")